In [ ]:
!pip install git+https://github.com/openai/whisper.git

Очищаем папку с получившимися видео предыдущего запуска

In [ ]:
!rm -rf /kaggle/working/* 

Функции для обработки одного видео, целой папки и подсчета итоговых фрагментов:

In [ ]:
import os
import numpy as np
from moviepy.editor import *
import whisper

def split_video_into_clips(video_path, output_path, min_duration=20, max_duration=40):
    audio_path = os.path.join(output_path, "temp_audio.wav")
    video = VideoFileClip(video_path)
    video.audio.write_audiofile(audio_path)

    model = whisper.load_model("small")
    transcription = model.transcribe(audio_path)

    clips = []
    current_clip = []
    current_clip_duration = 0
    for segment in transcription["segments"]:
        segment_duration = segment["end"] - segment["start"]
        if current_clip_duration + segment_duration > max_duration:
            clips.append(current_clip)
            current_clip = []
            current_clip_duration = 0
        current_clip.append(segment)
        current_clip_duration += segment_duration

    clips.append(current_clip)

    video = VideoFileClip(video_path)
    clip_counter = 1
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    
    for clip in clips:
        start_time = clip[0]["start"]
        end_time = clip[-1]["end"]
        clip_duration = end_time - start_time
        if clip_duration < min_duration:
            continue
        end_time = min(end_time, video.duration)
        clip_video = video.subclip(start_time, end_time)
        clip_filename = f"{video_name}_clip_{clip_counter}.mp4"
        clip_video.write_videofile(os.path.join(output_path, clip_filename))
        print(f"Сохранён клип {clip_counter} из видео '{video_name}' с длительностью {clip_duration:.2f} секунд.")
        clip_counter += 1

    video.close()
    os.remove(audio_path)  
    print(f"Видео '{video_name}' разбито на {clip_counter-1} клипов.\n")


def process_videos_folder(input_folder, output_folder, min_duration=20, max_duration=40):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"Создана выходная папка: {output_folder}")

    video_extensions = ('.mp4', '.avi', '.mov', '.mkv', '.webm', '.flv', '.m4v')

    video_files = []
    for file in os.listdir(input_folder):
        if file.lower().endswith(video_extensions):
            video_files.append(file)
    
    if not video_files:
        print(f"В папке '{input_folder}' не найдено видеофайлов.")
        return
    
    print(f"Найдено видеофайлов: {len(video_files)}")
    print("=" * 50)
    
    total_clips = 0
    
    for i, video_file in enumerate(video_files, 1):
        video_path = os.path.join(input_folder, video_file)
        print(f"\n[{i}/{len(video_files)}] Обработка: {video_file}")
        print("-" * 40)
        
        try:
            split_video_into_clips(video_path, output_folder, min_duration, max_duration)
            total_clips += get_clips_count(output_folder, video_file)
        except Exception as e:
            print(f"Ошибка при обработке видео '{video_file}': {str(e)}")
            continue
    
    print("\n" + "=" * 50)
    print(f"Обработка завершена, всего создано клипов: {total_clips}")
    print(f"Все клипы сохранены в папку: {output_folder}")


def get_clips_count(folder, video_name):
    video_basename = os.path.splitext(video_name)[0]
    count = 0
    for file in os.listdir(folder):
        if file.startswith(f"{video_basename}_clip_"):
            count += 1
    return count


# Примеры использования:

# 1. Обработка папки с видео
input_folder = "/kaggle/input/datasets/mariaspasyuk/videos-part4/downloads4"  # Папка с исходными видео
output_folder = "/kaggle/working/clips_output/"  # Папка для сохранения всех клипов

process_videos_folder(input_folder, output_folder, min_duration=20, max_duration=35)

# 2. Обработать одно видео
# video_path = ""
# output_path = "/kaggle/working/"
# split_video_into_clips(video_path, output_path)


Архивация папки с получившимися фрагментами и скачивание:

In [ ]:
import os
import zipfile
from IPython.display import FileLink

folder_to_zip = '/kaggle/working/clips_output'
zip_name = 'clips_output.zip'

def zip_folder(folder_path, output_path):
    if not os.path.exists(folder_path):
        print(f"Папка '{folder_path}' не найдена")
        return False

    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, start=os.path.dirname(folder_path))
                zipf.write(file_path, arcname)
    return True

print(f"Архивируем '{folder_to_zip}'")
if zip_folder(folder_to_zip, zip_name):
    file_size = os.path.getsize(zip_name) / (1024 * 1024)
    print(f"Архив создан: {zip_name} (Размер: {file_size:.2f} МБ)")
    
    print("Нажмите на ссылку ниже, чтобы скачать архив:")
    display(FileLink(zip_name))
else:
    print("Не удалось создать архив. Проверьте путь к папке.")